### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="splice",
    dataset_year="1991",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5M888",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/splice/ && wget -P local-data-warehouse/splice/ https://archive.ics.uci.edu/static/public/69/molecular+biology+splice+junction+gene+sequences.zip && unzip local-data-warehouse/splice/molecular+biology+splice+junction+gene+sequences.zip -d local-data-warehouse/splice/
""",
    # References
    academic_reference_bibtex="""@article{towell1994knowledge,
  title={Knowledge-based artificial neural networks},
  author={Towell, Geoffrey G and Shavlik, Jude W},
  journal={Artificial intelligence},
  volume={70},
  number={1-2},
  pages={119--165},
  year={1994},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="towell1994knowledge",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We tabularized the fixed-size DNA sequences from the original dataset based on the nucleotide position.
- We removed the instance ID.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SiteType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="SiteType",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/splice.data", index_col=False, header=None)

# Strip tab-like whitespaces from original data
df = df.map(lambda x: x.strip())
df = df.drop(columns=[1])  # Drop instance ID
target_feature = "SiteType"

split_data = df[2].apply(lambda x: pd.Series(list(x)))
# Generate new column names from -30 to +30 excluding 0
split_data.columns = [f"position_{i}" for i in range(-30, 31) if i != 0]
split_data[target_feature] = df[0]
df = split_data

cat_features = [f"position_{i}" for i in range(-30, 31) if i != 0] + [target_feature]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()